# Search-13 — Groupes automatiques : le problème du mot par automates

**Série Search · Part 1 — Fondations** · C# / .NET

[← Retour à la série](../README.md) · [Version Python](Search-13-AutomaticGroups-WordProblem.ipynb) · Précédent : [Search-12a — Composer des regards](Search-12a-Composer-Regards-CSharp.ipynb) · Suivant : [Search-13a — Traverser un mur](Search-13a-Traverser-Murs-Certifies-CSharp.ipynb)

Quand la structure du groupe donne l'automate, *décider* d'une propriété d'un mot devient un test
d'appartenance à un langage **régulier** — linéaire en la longueur du mot — au lieu d'une **exploration**
du graphe de Cayley. Ce carnet rend calculable cette frontière, sur les deux groupes types :
`Z^2` (croissance polynômiale) et `F_2` (croissance exponentielle).

## Enjeu — la heuristique ultime

Les carnets frères de la série posent le contraste exploration :

- **Search-02** (non informé) explore par **BFS** — il *parcourt* l'espace d'états ;
- **Search-03** (informé) explore par **A\*** — une heuristique *guide* le parcours, mais on explore toujours.

Un automate fini permet de décider si un mot appartient au langage qu'il reconnaît en une lecture de gauche à droite, en **O(|mot|)**, sans explorer le graphe de Cayley. Le DFA à six états construit ici reconnaît les **mots librement réduits** : ce sont les mots géodésiques de `F_2`, mais pas ceux de `Z^2` en général. Dans `Z^2`, la comparaison avec le BFS porte sur la famille `a^R`, dont les mots sont à la fois librement réduits et géodésiques.

## 1. Le cadre — deux groupes, un alphabet de 4 lettres

On travaille sur l'alphabet `{a, a⁻¹, b, b⁻¹}` (deux générateurs et leurs inverses).

- **`Z^2`** (libre abélien de rang 2) : un mot vaut l'identité si la *somme des exposants* de `a` est 0
  ET celle de `b` est 0. Le groupe est commutatif : `a b a⁻¹ b⁻¹ = 1`.
- **`F_2`** (groupe libre de rang 2) : un mot se réduit par *annulation libre* — on efface chaque lettre
  adjacente à son inverse. `a b a⁻¹` ne se réduit pas (le `b` bloque), mais `a a⁻¹` s'annule.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Alphabet : deux générateurs et leurs inverses. Un mot = LISTE de tokens.
var alphabet = new[] { "a", "a-1", "b", "b-1" };
var inverse = new Dictionary<string, string>(StringComparer.Ordinal)
{
    ["a"] = "a-1", ["a-1"] = "a", ["b"] = "b-1", ["b-1"] = "b"
};

// Analyse un mot compact : « aa-1 » donne les jetons « a » et « a-1 ».
List<string> Word(string text)
{
    var tokens = new List<string>();
    for (int i = 0; i < text.Length;)
    {
        int length = i + 2 < text.Length && text[i + 1] == '-' && text[i + 2] == '1' ? 3 : 1;
        tokens.Add(text.Substring(i, length));
        i += length;
    }
    return tokens;
}

// Image du mot dans Z^2 : somme des exposants de a et de b.
(int X, int Y) EvalZ2(IEnumerable<string> tokens)
{
    int x = 0, y = 0;
    foreach (var token in tokens)
    {
        switch (token)
        {
            case "a": x++; break;
            case "a-1": x--; break;
            case "b": y++; break;
            case "b-1": y--; break;
        }
    }
    return (x, y);
}

// Réduction libre dans F_2 : la fin de la liste sert de sommet de pile.
List<string> ReduceF2(IEnumerable<string> tokens)
{
    var reduced = new List<string>();
    foreach (var token in tokens)
    {
        if (reduced.Count > 0 && reduced[^1] == inverse[token])
            reduced.RemoveAt(reduced.Count - 1);
        else
            reduced.Add(token);
    }
    return reduced;
}

void Require(bool condition, string message)
{
    if (!condition)
        throw new InvalidOperationException(message);
}

// Comptabilise uniquement les comparaisons effectivement exécutées.
var parityBySection = new Dictionary<string, (int Comparisons, int Differences)>();
void CompareParity(string section, string item, string actual, string expected)
{
    parityBySection.TryGetValue(section, out var counts);
    bool equal = string.Equals(actual, expected, StringComparison.Ordinal);
    parityBySection[section] = (counts.Comparisons + 1, counts.Differences + (equal ? 0 : 1));
    Require(equal, $"ÉCART — {section}, {item} : C# = {actual} ; référence Python = {expected}.");
}

// Contrôles de base
var commutator = Word("aba-1b-1");
var image = EvalZ2(commutator); // a b a^{-1} b^{-1} = 1 dans Z^2 (commutatif)
var imageText = FormattableString.Invariant($"({image.X}, {image.Y})");
CompareParity("Mots", "image du commutateur", imageText, "(0, 0)"); // Sortie Python citée.
Require(ReduceF2(Word("aa-1")).Count == 0, "aa-1 doit se réduire au mot vide."); // annulation libre
var reducedExample = ReduceF2(Word("abb-1")); // b b^{-1} s'annule, reste 'a'
Require(reducedExample.SequenceEqual(Word("a")), "abb-1 doit se réduire à a.");
var reducedText = "[" + string.Join(", ", reducedExample.Select(token => $"'{token}'")) + "]";
CompareParity("Mots", "réduction de abb-1", reducedText, "['a']"); // Sortie Python citée.

Console.WriteLine($"Z^2 : a b a^-1 b^-1 vaut {imageText}");
Console.WriteLine($"F_2 : 'abb-1' se réduit en {reducedText}");

Z^2 : a b a^-1 b^-1 vaut (0, 0)


F_2 : 'abb-1' se réduit en ['a']


### Lecture du résultat

Dans `Z^2` le mot `a b a⁻¹ b⁻¹` vaut l'identité *par commutativité* (la somme des exposants est (0,0)),
alors que dans `F_2` le même enchaînement ne se réduit **pas** — la lettre `b` centrale bloque l'annulation
`a a⁻¹`. C'est toute la différence entre un groupe commutatif et un groupe libre : la *géométrie* des mots
n'y est pas du tout la même, et c'est précisément elle que les automates vont capturer.

## 2. Le problème du mot par automates — l'accepteur **construit**

Le langage des **mots librement réduits** est **régulier** : on le reconnaît par un
automate fini, et on le *construit* ici — l'état de l'automate n'est rien d'autre que **la dernière lettre
lue** (plus l'état de départ et un état mort pour les mots qui annulent).

Ces mots sont géodésiques dans `F_2`, mais pas nécessairement dans `Z^2` : pour être géodésique dans `Z^2`, un mot ne doit contenir aucun générateur avec ses deux signes, même si les occurrences ne sont pas adjacentes. Le libellé « Accepteur Z^2 » de la sortie Python est conservé pour la comparaison ; la table reconnaît les mots librement réduits.

In [2]:
// DFA à six états de la source : accepteur des mots librement réduits.
// Etats : S (depart) + un etat par derniere lettre + D (etat mort, mot qui s'annule).
var dfaStates = new[] { "S" }.Concat(alphabet).Append("D").ToArray();
var delta = dfaStates.ToDictionary(
    state => state,
    state => alphabet.ToDictionary(
        token => token,
        token => state == "D" || state == inverse[token] ? "D" : token,
        StringComparer.Ordinal),
    StringComparer.Ordinal);
var accepting = new HashSet<string>(dfaStates.Where(state => state != "D"), StringComparer.Ordinal); // tous les etats sauf l'etat mort D

// Lit chaque jeton une fois, en ne gardant que l'état courant et le compte des passages.
(string State, int Visits) RunAccepter(IEnumerable<string> tokens)
{
    string state = "S";
    int visits = 1; // Le passage par l'état initial compte aussi.
    foreach (var token in tokens)
    {
        state = delta[state][token];
        visits++;
    }
    return (state, visits);
}

bool IsFreelyReduced(IEnumerable<string> tokens) => accepting.Contains(RunAccepter(tokens).State);

// --- Verification PAR LEURS PROPRIETES (assertees, pas supposees) ---
// (1) un mot reduit est accepte
Require(IsFreelyReduced(Word("aba")), "aba doit être accepté.");
Require(IsFreelyReduced(Word("b")), "b doit être accepté.");
Require(IsFreelyReduced(Word("")), "Le mot vide doit être accepté."); // mot vide = geodesique de longueur 0

// (2) un mot avec annulation est rejete
Require(!IsFreelyReduced(Word("aa-1")), "aa-1 doit être rejeté.");
Require(!IsFreelyReduced(Word("ab-1b")), "ab-1b doit être rejeté."); // b-1 puis b s'annulent
Require(!IsFreelyReduced(Word("ba-1a")), "ba-1a doit être rejeté.");

// (3) Prolongement par un générateur : w reste librement réduit sauf si g annule la dernière lettre de w.
foreach (var compact in new[] { "a", "ab", "aba", "b-1", "ab-1" })
{
    var tokens = Word(compact);
    Require(IsFreelyReduced(tokens), $"Le préfixe {compact} doit être accepté.");
    foreach (var generator in alphabet)
    {
        bool expected = generator != inverse[tokens[^1]];
        Require(IsFreelyReduced(tokens.Append(generator)) == expected,
            $"Verdict de prolongement incorrect pour {compact} suivi de {generator}.");
    }
}

// D absorbe toute suite, même si celle-ci ne contient pas elle-même d'annulation.
Require(alphabet.All(token => delta["D"][token] == "D"), "L'état D doit être absorbant.");

// Le commutateur est accepté, mais sa distance à l'origine dans Z^2 vaut zéro.
var (commutatorX, commutatorY) = EvalZ2(commutator);
int commutatorDistance = Math.Abs(commutatorX) + Math.Abs(commutatorY);
Require(IsFreelyReduced(commutator) && ReduceF2(commutator).SequenceEqual(commutator)
    && commutatorDistance == 0 && commutator.Count > commutatorDistance,
    "Le commutateur doit être librement réduit sans être géodésique dans Z^2.");

// Référence citée : première ligne de la sortie Python de l'accepteur.
const string sourceAccepterSummary = "Accepteur Z^2 : 6 etats (S, a, a-1, b, b-1, D), 5 acceptants.";
string stateList = string.Join(", ", dfaStates.Where(delta.ContainsKey));
string accepterSummary = FormattableString.Invariant(
    $"Accepteur Z^2 : {delta.Count} etats ({stateList}), {accepting.Count} acceptants.");
CompareParity("Accepteur", "liste et effectifs des états", accepterSummary, sourceAccepterSummary);

Console.WriteLine(accepterSummary);
Console.WriteLine("Toutes les proprietes sont verifiees par les assertions ci-dessus.");

Accepteur Z^2 : 6 etats (S, a, a-1, b, b-1, D), 5 acceptants.


Toutes les proprietes sont verifiees par les assertions ci-dessus.


### Lecture du résultat

L'automate a **6 états** — bornés, indépendants de la longueur du mot. Il décide en une seule passe
gauche-droite, en **O(|mot|)** et en **mémoire constante** (un seul état courant). Tant qu'aucune
annulation adjacente n'a été détectée, cet état mémorise la dernière lettre lue après le départ ;
une fois atteint, l'état mort ne peut plus être quitté. Les trois propriétés sont vérifiées par les
appels à `Require` : un mot réduit est accepté, un mot qui annule est rejeté, et prolonger un mot
librement réduit par un générateur le laisse librement réduit, sauf quand ce générateur annule la
dernière lettre.

## 3. Les cônes géodésiques — la régularité, et l'automate qu'elle donne

Le **cône géodésique** de `Z^2` est l'ensemble des mots géodésiques ; dire que son langage est *régulier*
c'est dire qu'un automate fini le reconnaît. Dans `Z^2`, il suffit de mémoriser séparément, pour chaque
générateur, si aucun signe n'a été lu ou si le signe lu est positif ou négatif. Lire le signe opposé
pour un générateur déjà rencontré mène à un état mort.

Pour les mots librement réduits, l'automate ci-dessus **est** le témoin de régularité : sa table de
transition est finie et effective, pas une affirmation.

In [3]:
// La table de transition FINIE de l'accepteur des mots librement réduits : la regularite, effective.
// Références citées : les six lignes et le nombre d'états dans la sortie Python.
var sourceTransitionRows = new[]
{
    "    S |    a    a-1      b    b-1   <== acceptant",
    "    a |    a      D      b    b-1   <== acceptant",
    "  a-1 |    D    a-1      b    b-1   <== acceptant",
    "    b |    a    a-1      b      D   <== acceptant",
    "  b-1 |    a    a-1      D    b-1   <== acceptant",
    "    D |    D      D      D      D   (etat mort)"
};
CompareParity("Transitions", "nombre d'états", delta.Count.ToString(CultureInfo.InvariantCulture), "6");
Require(dfaStates.Length == sourceTransitionRows.Length, "Chaque état doit avoir une ligne de référence.");

// Les lignes affichées viennent de la table utilisée par RunAccepter, pas des références.
Console.WriteLine("  \\  | " + string.Join(" | ", alphabet.Select(token => token.PadLeft(4))));
Console.WriteLine(new string('-', 42));
for (int i = 0; i < dfaStates.Length; i++)
{
    string state = dfaStates[i];
    string row = string.Join("   ", alphabet.Select(token => delta[state][token].PadLeft(4)));
    string mark = accepting.Contains(state) ? "   <== acceptant" : state == "D" ? "   (etat mort)" : "";
    string rendered = $"  {state,3} | {row}{mark}";
    CompareParity("Transitions", state, rendered, sourceTransitionRows[i]);
    Console.WriteLine(rendered);
}
Console.WriteLine();
Console.WriteLine(FormattableString.Invariant($"Nombre d'etats : {delta.Count} (borné, indépendant de la longueur des mots)."));

  \  |    a |  a-1 |    b |  b-1


------------------------------------------


    S |    a    a-1      b    b-1   <== acceptant


    a |    a      D      b    b-1   <== acceptant


  a-1 |    D    a-1      b    b-1   <== acceptant


    b |    a    a-1      b      D   <== acceptant


  b-1 |    a    a-1      D    b-1   <== acceptant


    D |    D      D      D      D   (etat mort)


Nombre d'etats : 6 (borné, indépendant de la longueur des mots).


## 4. Le contraste qui compte — BFS (explorer) vs accepteur (décider)

Pour la famille de mots géodésiques `a^R` de longueur `R`, on compare deux coûts :

- **BFS** sur le graphe de Cayley de `Z^2` : on explore la boule de rayon `R`, qui compte **Θ(R²)**
  noeuds — on *explore* pour décider ;
- **accepteur** : on lit le mot en **O(|mot|) = O(R)**, *sans explorer*.

Le rapport *noeuds-BFS / états-accepteur* sur des sphères croissantes fait voir l'écart se creuser.
On compte les passages par les états de l'accepteur, état initial compris, et non les états distincts.
Sur `a^R`, l'absence d'annulation donne le même verdict que le critère géodésique de `Z^2`.

In [4]:
// BFS dans le graphe de Cayley de Z² et lecture de a^R.
int BfsBallZ2(int radius)
{
    var directions = new (int Dx, int Dy)[] { (1, 0), (-1, 0), (0, 1), (0, -1) };
    var seen = new HashSet<(int X, int Y)> { (0, 0) };
    var queue = new Queue<(int X, int Y, int Distance)>();
    queue.Enqueue((0, 0, 0));
    while (queue.Count > 0)
    {
        var (x, y, distance) = queue.Dequeue();
        if (distance == radius)
            continue;
        foreach (var (dx, dy) in directions)
        {
            var neighbor = (X: x + dx, Y: y + dy);
            if (seen.Add(neighbor))
                queue.Enqueue((neighbor.X, neighbor.Y, distance + 1));
        }
    }
    return seen.Count;
}

// Références citées : rayon, nœuds, passages et rapport affichés dans la sortie Python.
var sourceBfsRows = new (int Radius, int Nodes, int Visits, string Ratio)[]
{
    (1, 5, 2, "2.5"), (2, 13, 3, "4.3"), (3, 25, 4, "6.2"), (5, 61, 6, "10.2"),
    (8, 145, 9, "16.1"), (12, 313, 13, "24.1"), (20, 841, 21, "40.0"), (30, 1861, 31, "60.0")
};

// Un mot geodesique w_R = a^R de longueur R : BFS explore la boule, l'accepteur lit le mot.
Console.WriteLine("   R  |  BFS noeuds  |  accepteur etats  |  rapport BFS/accepteur");
Console.WriteLine("      |              |                   |" + new string(' ', 8));
foreach (var (radius, expectedNodes, expectedVisits, expectedRatio) in sourceBfsRows)
{
    int nodes = BfsBallZ2(radius); // ~ 2R^2 + 2R + 1  (quadratique)
    var rayWord = Word(new string('a', radius));
    var (state, visits) = RunAccepter(rayWord); // |w_R| + 1 etats parcourus (linéaire)
    var (x, y) = EvalZ2(rayWord);
    Require(accepting.Contains(state) && Math.Abs(x) + Math.Abs(y) == rayWord.Count,
        "Le mot a^R doit être accepté et géodésique dans Z^2.");
    Require(nodes == 2 * radius * radius + 2 * radius + 1 && visits == rayWord.Count + 1,
        "Les coûts mesurés doivent satisfaire les formules annoncées.");

    // Arrondi au pair au dixième comme en Python : 25/4 = 6.25 donne 6.2.
    string ratio = decimal.Round((decimal)nodes / visits, 1, MidpointRounding.ToEven)
        .ToString("F1", CultureInfo.InvariantCulture);
    string testCase = FormattableString.Invariant($"R = {radius}");
    CompareParity("BFS / accepteur", testCase + " : nœuds", nodes.ToString(CultureInfo.InvariantCulture),
        expectedNodes.ToString(CultureInfo.InvariantCulture));
    CompareParity("BFS / accepteur", testCase + " : passages", visits.ToString(CultureInfo.InvariantCulture),
        expectedVisits.ToString(CultureInfo.InvariantCulture));
    CompareParity("BFS / accepteur", testCase + " : rapport au dixième", ratio, expectedRatio);
    Console.WriteLine(FormattableString.Invariant($"  {radius,3}  |  {nodes,9}  |  {visits,15}  |  {ratio,8} x"));
}
Console.WriteLine("\nLe BFS croit en ~2R^2 (quadratique), l'accepteur en R (linéaire) : le rapport diverge en ~2R.");

   R  |  BFS noeuds  |  accepteur etats  |  rapport BFS/accepteur


      |              |                   |        


    1  |          5  |                2  |       2.5 x


    2  |         13  |                3  |       4.3 x


    3  |         25  |                4  |       6.2 x


    5  |         61  |                6  |      10.2 x


    8  |        145  |                9  |      16.1 x


   12  |        313  |               13  |      24.1 x


   20  |        841  |               21  |      40.0 x


   30  |       1861  |               31  |      60.0 x



Le BFS croit en ~2R^2 (quadratique), l'accepteur en R (linéaire) : le rapport diverge en ~2R.


### Lecture du résultat

Le BFS visite `2R² + 2R + 1` noeuds pour couvrir la boule de rayon `R` (d'où le `~2R²`), tandis que
l'accepteur ne parcourt que `R + 1` états, comptés avec répétition. Le rapport `BFS/accepteur` croît donc **linéairement avec R**
(colonne de droite) : à `R = 30` le BFS explore ~60 fois plus d'objets que l'automate n'en lit. C'est la
frontière « décider vs explorer » faite chiffre — la structure du groupe rend l'exploration *inutile*.

## 5. La croissance — polynôme (Z²) vs exponentielle (F₂), **comptée**

La signature d'un groupe automatique se lit dans la **croissance** de sa boule. On ne *cite* pas les
formules : on **compte** les sphères et les boules, pour `k = 0..8`, en balayant les coordonnées dans `Z^2`
et en parcourant l'arbre de Cayley de `F_2` par niveaux.

In [5]:
// Comptage des sphères et des boules de Z² et de F₂.
int SphereZ2(int k)
{
    // À distance EXACTEMENT k : |x| + |y| = k.
    int count = 0;
    for (int x = -k; x <= k; x++)
    {
        int remaining = k - Math.Abs(x);
        count += remaining == 0 ? 1 : 2;
    }
    return count;
}

int BallZ2(int k) => Enumerable.Range(0, k + 1).Sum(SphereZ2);

int SphereF2(int k)
{
    // Le niveau initial ne contient que le mot vide ; chaque mot = liste de tokens.
    var level = new List<List<string>> { new List<string>() };
    for (int depth = 0; depth < k; depth++)
    {
        var nextLevel = new List<List<string>>();
        foreach (var tokens in level)
        {
            foreach (var token in alphabet)
            {
                if (tokens.Count > 0 && token == inverse[tokens[^1]])
                    continue;
                var extension = new List<string>(tokens) { token };
                nextLevel.Add(extension);
            }
        }
        level = nextLevel;
    }
    return level.Count;
}

int BallF2(int k) => Enumerable.Range(0, k + 1).Sum(SphereF2);

// Références citées : les quatre colonnes de la sortie Python, dans l'ordre k = 0..8.
var sourceGrowthRows = new (int SphereZ, int BallZ, int SphereF, int BallF)[]
{
    (1, 1, 1, 1), (4, 5, 4, 5), (8, 13, 12, 17),
    (12, 25, 36, 53), (16, 41, 108, 161), (20, 61, 324, 485),
    (24, 85, 972, 1457), (28, 113, 2916, 4373), (32, 145, 8748, 13121)
};

// La signature : Z^2 quadratique (2e difference finie constante), F_2 exponentielle (ratio ~3).
var measuredBallsZ2 = new List<int>();
var measuredSpheresF2 = new List<int>();
int powerOfThree = 1;
Console.WriteLine("  k  |  Z^2 sphere |  Z^2 boule  |   F2 sphere  |   F2 boule");
Console.WriteLine("     |            |             |              |");
for (int k = 0; k <= 8; k++)
{
    int sphereZ = SphereZ2(k), ballZ = BallZ2(k);
    int sphereF = SphereF2(k), ballF = BallF2(k);
    var expected = sourceGrowthRows[k];
    string testCase = FormattableString.Invariant($"k = {k}");
    CompareParity("Croissance", testCase + " : sphère Z²", sphereZ.ToString(CultureInfo.InvariantCulture),
        expected.SphereZ.ToString(CultureInfo.InvariantCulture));
    CompareParity("Croissance", testCase + " : boule Z²", ballZ.ToString(CultureInfo.InvariantCulture),
        expected.BallZ.ToString(CultureInfo.InvariantCulture));
    CompareParity("Croissance", testCase + " : sphère F₂", sphereF.ToString(CultureInfo.InvariantCulture),
        expected.SphereF.ToString(CultureInfo.InvariantCulture));
    CompareParity("Croissance", testCase + " : boule F₂", ballF.ToString(CultureInfo.InvariantCulture),
        expected.BallF.ToString(CultureInfo.InvariantCulture));

    Require(ballZ == 2 * k * k + 2 * k + 1 && ballF == 2 * powerOfThree - 1,
        testCase + " : les comptages doivent satisfaire les formules des boules.");
    if (k >= 2)
    {
        Require(ballZ - 2 * measuredBallsZ2[k - 1] + measuredBallsZ2[k - 2] == 4,
            testCase + " : la deuxième différence des boules de Z² doit valoir 4.");
        Require(sphereF == 3 * measuredSpheresF2[k - 1],
            testCase + " : la sphère de F₂ doit tripler après le premier niveau.");
    }
    measuredBallsZ2.Add(ballZ);
    measuredSpheresF2.Add(sphereF);
    powerOfThree *= 3;
    Console.WriteLine(FormattableString.Invariant($"  {k,2}  |  {sphereZ,8}  |  {ballZ,8}  |  {sphereF,10}  |  {ballF,10}"));
}

Require(BallZ2(10) == 2 * 10 * 10 + 2 * 10 + 1, "La formule de la boule de Z² doit tenir en k = 10."); // 221 : croissance polynomiale (degre 2)
Require(BallF2(5) == 2 * (3 * 3 * 3 * 3 * 3) - 1, "La formule de la boule de F₂ doit tenir en k = 5."); // 485 : croissance exponentielle (base 3)
Console.WriteLine("\nZ^2 : boule = 2k^2 + 2k + 1  (polynome, degre 2).   F_2 : boule = 2·3^k - 1  (exponentielle, base 3).");

// Bilan alimenté par les comparaisons réellement exécutées avec les sorties Python citées.
Console.WriteLine("\nAttestation de parité — résultats C# comparés aux références Python citées");
foreach (var section in new[] { "Mots", "Accepteur", "Transitions", "BFS / accepteur", "Croissance" })
{
    Require(parityBySection.TryGetValue(section, out var result) && result.Comparisons > 0,
        $"Aucune comparaison pour la section {section}.");
    Require(result.Differences == 0, $"Écart de parité dans la section {section}.");
    Console.WriteLine(FormattableString.Invariant(
        $"{section} : {result.Differences} écart(s) sur {result.Comparisons} comparaison(s) — CONFORME"));
}

  k  |  Z^2 sphere |  Z^2 boule  |   F2 sphere  |   F2 boule


     |            |             |              |


   0  |         1  |         1  |           1  |           1


   1  |         4  |         5  |           4  |           5


   2  |         8  |        13  |          12  |          17


   3  |        12  |        25  |          36  |          53


... (8 outputs left out)

Accepteur : 0 écart(s) sur 1 comparaison(s) — CONFORME


Transitions : 0 écart(s) sur 7 comparaison(s) — CONFORME


BFS / accepteur : 0 écart(s) sur 24 comparaison(s) — CONFORME


Croissance : 0 écart(s) sur 36 comparaison(s) — CONFORME


### Lecture du résultat

La table fait la différence sans l'affirmer : la boule de `Z^2` suit `2k² + 2k + 1` (les *deuxièmes
différences finies* sont constantes — signature d'une croissance **polynomiale de degré 2**), alors que
celle de `F_2` suit `2·3^k − 1` (le *ratio* d'une boule à la précédente tend vers 3 — signature d'une
croissance **exponentielle**). C'est exactement le contraste que la théorie des groupes automatiques met
en scène : `Z^2` est automatique à croissance polynomiale, `F_2` est automatique à croissance
exponentielle — et les deux admettent un automate fini de mots géodésiques.

## Synthèse — où se lit la frontière « décider / explorer »

| Question | Sans structure (Search-02/03) | Avec l'automate du groupe (ce carnet) |
|---|---|---|
| « ce mot est-il géodésique ? » | BFS jusqu'au rayon \|mot\| — **Θ(\|mot\|²)** noeuds | lecture gauche-droite du DFA — **O(\|mot\|)**, mémoire constante |
| Coût en croissance | explore la boule (polynôme pour Z², exponentiel pour F₂) | indépendant de la boule — linéaire en le mot |

Quand le groupe *fournit* l'automate, la géométrie qui coûtait une exploration entière se réduit à un
test d'appartenance à un langage régulier. C'est le sens du « word processing » : le groupe fait le travail
de l'heuristique.

Le coût quadratique du tableau concerne `Z^2`. Pour la question de géodésicité, la colonne de droite suppose un DFA qui reconnaît les géodésiques du groupe considéré : dans `Z^2`, le DFA à six états de la mesure donne le bon verdict sur `a^R`, mais pas sur tous les mots. Reconnaître les géodésiques et décider si un mot vaut l'identité sont deux questions distinctes.

## Exercices

In [6]:
// Illustration 1 — Exercice 1 : L'accepteur géodésique de Z^3
// Construire un DFA qui reconnaît exactement les mots géodésiques de Z^3
// sur l'alphabet {a, a-1, b, b-1, c, c-1}. Combien d'états faut-il ?
Dictionary<string, Dictionary<string, string>> GeodesicAccepterZ3()
{
    // TODO étudiant
    return new Dictionary<string, Dictionary<string, string>>(); // TODO étudiant
}

In [7]:
// Illustration 2 — Exercice 2 : Le ratio BFS / accepteur sur F_2
// Pour un mot géodésique w de longueur R dans F_2, mesurer le rapport entre les
// nœuds explorés par un BFS sur le graphe de Cayley de F_2 et les passages par
// les états de l'accepteur. Décrire comment ce ratio croît avec R et le relier
// à la croissance exponentielle de F_2 (section 5).
double RatioBfsAccepteurF2(int radius)
{
    // TODO étudiant
    return 0.0; // TODO étudiant
}

In [8]:
// Illustration 3 — Exercice 3 : Le problème du mot COMPLET de Z^2 n'est pas régulier
// Le langage des mots qui valent l'identité dans Z^2 (cf. EvalZ2) n'est pas
// régulier. Établir ce résultat à l'aide du lemme de la pompe sur les mots
// équilibrés de l'alphabet {a, a-1}. Pourquoi cette restriction ne contredit-elle
// pas la régularité du cône géodésique ?
string PourquoiPasRegulier()
{
    // TODO étudiant
    return string.Empty; // TODO étudiant
}

## À retenir

- Le langage des mots géodésiques de `Z^2` est **régulier**. L'automate à **6 états** construit ici reconnaît les mots librement réduits, qui sont les géodésiques de `F_2` (sections 2–3).
- Sur les mots `a^R` de la comparaison, la lecture par l'accepteur coûte **O(R)** ; le BFS dans `Z^2` explore **Θ(R²)** nœuds — l'écart diverge (section 4).
- `Z^2` croît **polynomialement** (`2k²+2k+1`), `F_2` **exponentiellement** (`2·3^k−1`) — compté, pas cité
  (section 5).
- La structure algébrique du groupe, quand elle fournit l'automate, *remplace* l'exploration par une
  décision : c'est le fil qui relie ce carnet à Search-02/03.

### Attestation de parité

Le relevé à la fin du calcul de croissance donne, par section, le nombre de comparaisons exécutées et les écarts entre les résultats C# et les références citées de la sortie Python. La comparaison conserve les transitions de la source sans identifier les mots librement réduits aux géodésiques de `Z^2`.

## Bibliographie

- D. Epstein, J. W. Cannon, D. D. Holt, S. Levy, K. C. Paterson, W. P. Thurston, *Word Processing in
  Groups*, 1992 — le « word processing » et les cônes géodésiques.
- M. Gromov, *Groups that act on hyperbolic spaces*, Invent. Math. 1987 — structures automatiques.
- Carnets frères : [Search-02 — Recherche non informée](Search-02-Uninformed-CSharp.ipynb) (BFS) ·
  [Search-03 — Recherche informée](Search-03-Informed-CSharp.ipynb) (A*).

> Provenance : [version Python](Search-13-AutomaticGroups-WordProblem.ipynb), source des instances et des sorties de référence citées.